<a href="https://colab.research.google.com/github/AnjaliKM-S/ICT-AKM_2026/blob/main/Casestudy_and_Intermediate_Assessment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##LOADED DATA FROM BOTH API

In [5]:
import requests

#API 1: Launches
url1 = "https://api.spacexdata.com/v4/launches"

#API 2: Rockets
url2 = "https://api.spacexdata.com/v4/rockets"

try:
    response1 = requests.get(url1)
    response1.raise_for_status()
    launches = response1.json()

    response2 = requests.get(url2)
    response2.raise_for_status()
    rockets = response2.json()

    print("Both API calls successful")

except requests.exceptions.RequestException as e:
    print(f"API Call Failed: {e}")

Both API calls successful


###SpaceX Launch Data from API
####Extract relevant columns: name, date_utc, success, details, rocket
####Convert date_utc to datetime and extract the year




In [8]:
import pandas as pd

df_launch = pd.DataFrame(launches)

# columns
df_launch = df_launch[["name", "date_utc", "success", "details", "rocket"]]

# Convert date_utc to datetime
df_launch["date_utc"] = pd.to_datetime(df_launch["date_utc"], errors="coerce")

# Extract year
df_launch["year"] = df_launch["date_utc"].dt.year


print(df_launch.head())


          name                  date_utc success  \
0    FalconSat 2006-03-24 22:30:00+00:00   False   
1      DemoSat 2007-03-21 01:10:00+00:00   False   
2  Trailblazer 2008-08-03 03:34:00+00:00   False   
3       RatSat 2008-09-28 23:15:00+00:00    True   
4     RazakSat 2009-07-13 03:35:00+00:00    True   

                                             details  \
0   Engine failure at 33 seconds and loss of vehicle   
1  Successful first stage burn and transition to ...   
2  Residual stage 1 thrust led to collision betwe...   
3  Ratsat was carried to orbit on the first succe...   
4                                               None   

                     rocket  year  
0  5e9d0d95eda69955f709d1eb  2006  
1  5e9d0d95eda69955f709d1eb  2007  
2  5e9d0d95eda69955f709d1eb  2008  
3  5e9d0d95eda69955f709d1eb  2008  
4  5e9d0d95eda69955f709d1eb  2009  


##Rocket Metadata
####Extract id, name, type, active, and stages


In [57]:
import pandas as pd

df_rocket = pd.DataFrame(rockets)

# columns
df_rocket = df_rocket[["id", "name","type", "active", "stages"]]
df_rocket.rename(columns={"name": "rocket_name"}, inplace=True)


print(df_rocket.head())

                         id   rocket_name    type  active  stages
0  5e9d0d95eda69955f709d1eb      Falcon 1  rocket   False       2
1  5e9d0d95eda69973a809d1ec      Falcon 9  rocket    True       2
2  5e9d0d95eda69974db09d1ed  Falcon Heavy  rocket    True       2
3  5e9d0d96eda699382d09d1ee      Starship  rocket   False       2


##Step 3: Merge Launch and Rocket Data
####● Join the two DataFrames on rocket ID using


In [58]:
merged_df = pd.merge(df_launch,
    df_rocket,
    left_on="rocket",
    right_on="id")

print(merged_df.head())

          name                  date_utc success  \
0    FalconSat 2006-03-24 22:30:00+00:00   False   
1      DemoSat 2007-03-21 01:10:00+00:00   False   
2  Trailblazer 2008-08-03 03:34:00+00:00   False   
3       RatSat 2008-09-28 23:15:00+00:00    True   
4     RazakSat 2009-07-13 03:35:00+00:00    True   

                                             details  \
0   Engine failure at 33 seconds and loss of vehicle   
1  Successful first stage burn and transition to ...   
2  Residual stage 1 thrust led to collision betwe...   
3  Ratsat was carried to orbit on the first succe...   
4                                               None   

                     rocket  year                        id rocket_name  \
0  5e9d0d95eda69955f709d1eb  2006  5e9d0d95eda69955f709d1eb    Falcon 1   
1  5e9d0d95eda69955f709d1eb  2007  5e9d0d95eda69955f709d1eb    Falcon 1   
2  5e9d0d95eda69955f709d1eb  2008  5e9d0d95eda69955f709d1eb    Falcon 1   
3  5e9d0d95eda69955f709d1eb  2008  5e9d0d95eda6995

###Add Simulated Country Information
####new column country and randomly assign one of these values:
#####['USA', 'Russia', 'India', 'China', 'France']


In [59]:
countries = ['USA', 'Russia', 'India', 'China', 'France']
merged_df["country"] = pd.Series(countries).sample(len(merged_df), replace=True).values

print(merged_df)

                       name                  date_utc success  \
0                 FalconSat 2006-03-24 22:30:00+00:00   False   
1                   DemoSat 2007-03-21 01:10:00+00:00   False   
2               Trailblazer 2008-08-03 03:34:00+00:00   False   
3                    RatSat 2008-09-28 23:15:00+00:00    True   
4                  RazakSat 2009-07-13 03:35:00+00:00    True   
..                      ...                       ...     ...   
200           Transporter-6 2022-12-01 00:00:00+00:00    None   
201                   TTL-1 2022-12-01 00:00:00+00:00    None   
202  WorldView Legion 1 & 2 2022-12-01 00:00:00+00:00    None   
203     Viasat-3 & Arcturus 2022-12-01 00:00:00+00:00    None   
204          O3b mPower 3.4 2022-12-01 00:00:00+00:00    None   

                                               details  \
0     Engine failure at 33 seconds and loss of vehicle   
1    Successful first stage burn and transition to ...   
2    Residual stage 1 thrust led to collision

###Store Merged Data in SQLite3
● Use sqlite3 to create a connection and save the merged DataFrame as a table named
launches

● Table should contain all merged columns including country

In [60]:
import sqlite3

In [61]:
def load_spacex_database(db_path):
  try:
    conn = sqlite3.connect("spacex.db")
    merged_df.to_sql("launches", conn, if_exists="replace", index=False)

    print("spacex database loaded successfully.")
    return conn
  except sqlite3.Error as e:
    print(f"Error loading spacex database: {e}")
    return None

In [62]:
db_path = "/content/spacex.db"
conn = load_spacex_database(db_path)

if conn:
  print("Connection successful")
  cursor = conn.cursor()

spacex database loaded successfully.
Connection successful


In [63]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("Database Schema:")
for table in tables:
    table_name = table[0]
    print(f"\nTable: {table_name}")

    cursor.execute(f"PRAGMA table_info({table_name});")
    schema = cursor.fetchall()

    print("Columns:")
    for column in schema:
        cid, name, dtype, notnull, dflt_value, pk = column
        print(f"  - {name} ({dtype}), NOT NULL: {bool(notnull)}, Default: {dflt_value}, Primary Key: {bool(pk)}")

Database Schema:

Table: launches
Columns:
  - name (TEXT), NOT NULL: False, Default: None, Primary Key: False
  - date_utc (TIMESTAMP), NOT NULL: False, Default: None, Primary Key: False
  - success (INTEGER), NOT NULL: False, Default: None, Primary Key: False
  - details (TEXT), NOT NULL: False, Default: None, Primary Key: False
  - rocket (TEXT), NOT NULL: False, Default: None, Primary Key: False
  - year (INTEGER), NOT NULL: False, Default: None, Primary Key: False
  - id (TEXT), NOT NULL: False, Default: None, Primary Key: False
  - rocket_name (TEXT), NOT NULL: False, Default: None, Primary Key: False
  - type (TEXT), NOT NULL: False, Default: None, Primary Key: False
  - active (INTEGER), NOT NULL: False, Default: None, Primary Key: False
  - stages (INTEGER), NOT NULL: False, Default: None, Primary Key: False
  - country (TEXT), NOT NULL: False, Default: None, Primary Key: False


#### Run SQL Queries on the Data to analyze
1. Launches by Country
2. Which year had the highest number of launches?
3. Top 5 Missions by Launch Count

####1.Launches by Country

In [79]:
#q1 ="SELECT country FROM Launches ORDER BY country;"
q1 ="SELECT country, COUNT(*) AS total_launches FROM launches GROUP BY country ORDER BY total_launches DESC;"
result = pd.read_sql_query(q1,conn)
result


,country,total_launches
0,France,50
1,Russia,45
2,India,37
3,China,37
4,USA,36


####year had the highest number of launches

In [67]:
q2 ="SELECT year, COUNT(*) AS total_launches FROM launches GROUP BY year ORDER BY total_launches DESC LIMIT 1;"
result = pd.read_sql_query(q2,conn)
result

,year,total_launches
0,2022,62


####Top 5 Missions by Launch Count

In [68]:
q3 = "select * FROM launches;"
result = pd.read_sql_query(q3,conn)
result

,name,date_utc,success,details,rocket,year,id,rocket_name,type,active,stages,country
0,FalconSat,2006-03-24 22:30:00+00:00,0.0,Engine failure at 33 seconds and loss of vehicle,5e9d0d95eda69955f709d1eb,2006,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,0,2,USA
1,DemoSat,2007-03-21 01:10:00+00:00,0.0,Successful first stage burn and transition to ...,5e9d0d95eda69955f709d1eb,2007,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,0,2,India
2,Trailblazer,2008-08-03 03:34:00+00:00,0.0,Residual stage 1 thrust led to collision betwe...,5e9d0d95eda69955f709d1eb,2008,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,0,2,Russia
3,RatSat,2008-09-28 23:15:00+00:00,1.0,Ratsat was carried to orbit on the first succe...,5e9d0d95eda69955f709d1eb,2008,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,0,2,Russia
4,RazakSat,2009-07-13 03:35:00+00:00,1.0,None,5e9d0d95eda69955f709d1eb,2009,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,0,2,India
...,...,...,...,...,...,...,...,...,...,...,...,...
200,Transporter-6,2022-12-01 00:00:00+00:00,NaN,None,5e9d0d95eda69973a809d1ec,2022,5e9d0d95eda69973a809d1ec,Falcon 9,rocket,1,2,USA
201,TTL-1,2022-12-01 00:00:00+00:00,NaN,None,5e9d0d95eda69973a809d1ec,2022,5e9d0d95eda69973a809d1ec,Falcon 9,rocket,1,2,USA
202,WorldView Legion 1 & 2,2022-12-01 00:00:00+00:00,NaN,None,5e9d0d95eda69973a809d1ec,2022,5e9d0d95eda69973a809d1ec,Falcon 9,rocket,1,2,France
203,Viasat-3 & Arcturus,2022-12-01 00:00:00+00:00,NaN,None,5e9d0d95eda69974db09d1ed,2022,5e9d0d95eda69974db09d1ed,Falcon Heavy,rocket,1,2,France


In [71]:
q3 = "SELECT name, COUNT(*) AS launch_count FROM launches GROUP BY name ORDER BY launch_count DESC LIMIT 5;"
result = pd.read_sql_query(q3,conn)
result

,name,launch_count
0,ispace Mission 1 & Rashid,1
1,ZUMA,1
2,WorldView Legion 1 & 2,1
3,Viasat-3 & Arcturus,1
4,USSF-44,1


In [80]:
q3="SELECT name, AVG(success) AS success_rate FROM launches GROUP BY name ORDER BY success_rate DESC LIMIT 5;"
pd.read_sql_query(q3,conn)

,name,success_rate
0,ZUMA,1.0
1,Türksat 5B,1.0
2,TürkmenÄlem 52°E / MonacoSAT,1.0
3,Turksat 5A,1.0
4,Transporter-5,1.0
